# Create and Upload I14Y Concept

## First Creation

If there is no concept yet on I14Y with the static identifier `political-parties`

- create concept entries from `parties.json` dictionary
- create a new concept based on `concept_teamplate.json` via API
- upload the concept entries to the newly created concept via API
- set the concept to public

## Update Existing Concept

- create concept entries from `parties.json` dictionary
- get the versioned identifier of the most current concept via API
- check if the newly locally created concept entries differ from the existing concept on I14Y, if not --> end, else:
- set the validTo value of the existing concept to the current date
- create a new version of the concept (with the same identifier `political-parties` and a new semantic version)
- upload the concept entries to the newly created concept via API
- set the new concept to public

## I14Y Learnings

- if the concept is not public, it will not be visible in the I14Y search results via GUI and also not via API
- if a single field of the concept needs to be updated, the whole concept needs to be downloaded, updated and uploaded again, as there is no partial update available via API

## Variable Naming

The concepts on I14Y have two different identifiers:

- static identifier:
  - does not change between versions
  - key in I14Y: `identifiers`
  - variable name in this notebook: `static_identifier`
  - value in this notebook: `political-parties`
- versioned identifier:
  - changes with each new version
  - is a UUID generated by I14Y and can not be set by the user
  - key in I14Y: `conceptId`
  - variable name in this notebook: `versioned_identifier`
  - value in this notebook: e.g. `01a120af-e345-72be-888a-4e1ebb918106`

Naming for concept and concept entries:

- `xy_concept`: this is a dictionary containing the concept key-value pairs, be it loaded from the template or downloaded from I14Y via API
- `xy_concept_entries`: this is a dictionary containing the concept entries key-value pairs, be it created from the local `parties.json` or downloaded from I14Y via API

## Import Libraries

In [1]:
import os
import copy
import json
import pandas as pd
import requests
from dotenv import load_dotenv
from datetime import datetime

## Function Definitions

In [2]:
def create_concept_entries_from_json(parties):
    """
    Create concept entries from the parties JSON data.

    Args:
        parties (dict): The parties JSON data.

    Returns:
        dict: The new concept entries.
    """
    
    # base für codes json
    entries = {
        "data": [],
    }

    for id, party in parties.items():

        # only identity is relevant
        party = party.get("identity")

        if party.get("name") and pd.isna(party.get("dissolution_date")):  # check if the party has a name and is not deprecated
            code_entry = {
                "annotations": [
                    {
                        "type": "EXT_RESOURCE",
                        "uri": f"https://politics.ld.admin.ch/political-party/{id}"
                    }
                ],
                "code": str(id),  # get the id of the party
                "name": party.get("name")
            }
            # if the party has a parent, add the parent code to the code entry
            child_of = [relation for relation in party.get("relationships", []) if relation.get("type_label") == "child-of"]
            if len(child_of) > 0:
                child_of = child_of[0]  # get the first child_of relation
                code_entry["parentCode"] = str(child_of.get("party_to"))
            entries["data"].append(code_entry)

    return entries

In [3]:
def get_current_concept(static_identifier):
    """
    Get the latest concept from the i14y API.

    Args:
        static_identifier (str): The static identifier of the concept that never changes, e.g. "political-parties".

    Returns:
        dict: The latest concept.
    """
    
    url = f"https://api-a.i14y.admin.ch/api/public/v1/concepts/"

    params = {
        'conceptIdentifier': static_identifier, 
    }

    try:
        response = requests.get(url, params=params, verify = False, timeout=10)
        response.raise_for_status()  # Raise an exception for HTTP errors
        current_concept = response.json()
    except requests.exceptions.HTTPError as e:
        print(f"HTTP-Error {e.response.status_code}: {e.response.text[:200]}")
    except requests.exceptions.Timeout:
        print("Timeout")

    print(f"Current concept: {current_concept}")


    # sort data by 'version' in descending order
    current_concept = sorted(current_concept["data"], key=lambda x: x['version'], reverse=True)
    current_concept = {"data": current_concept[0]}

    return current_concept

In [4]:
def get_current_concept_entries(versioned_identifier):
    """
    Get the latest concept entries from the i14y API.

    Args:
        versioned_identifier (str): The versioned identifier of the concept, e.g. "01a120af-e345-72be-888a-4e1ebb918106".


    Returns:
        dict: The latest concept entries.
    """
    
    url = f"https://api-a.i14y.admin.ch/api/public/v1/concepts/{versioned_identifier}/codelist-entries/exports/json"

    response = requests.get(url, verify = False)

    if response.status_code == 200:
        current_concept_entries = json.loads(response.content)
    else:
        print(f"Failed to get data, status code: {response.status_code}")

    return current_concept_entries

In [5]:
def compare_concept_entries(current_concept_entries, new_concept_entries):
    """
    Compare the current concept entries with the new concept entries.

    Args:
        current_concept_entries (dict): The current concept entries.
        new_concept_entries (dict): The new concept entries.

    Returns:
        bool: True if there are changes, False otherwise.
    """
    
    current_codes = set(json.dumps(d, sort_keys=True) for d in current_concept_entries["data"])
    new_codes = set(json.dumps(d, sort_keys=True) for d in new_concept_entries["data"])

    if current_codes == new_codes:
        print("No changes detected. The code list is up to date.")
        return False
    else:
        print("Changes detected. The code list needs to be updated.")
        return True

In [6]:
def get_access_token(client_id, client_secret):
    """
    Get an access token from the i14y identity provider.

    Args:
        client_id (str): The client ID.
        client_secret (str): The client secret.
    
    Returns:
        str: The access token.
    """
    data = {'grant_type': 'client_credentials'}
    response = requests.post("https://identity.i14y.a.c.bfs.admin.ch/realms/bfs-sis-a/protocol/openid-connect/token", data=data, verify=False, auth=(client_id, client_secret))
    data = response.json()
    return 'Bearer ' + data['access_token']

In [7]:
def set_concept_end_date(current_concept, concept_template, end_date, access_token):
    """
    Set the end date of a concept in the i14y API.

    Args:
        current_concept (dict): The current concept.
        concept_template (dict): The concept template.
        end_date (str): The end date in isoformat, e.g. "2024-06-30T00:00:00".
        access_token (str): The access token.
    
    Returns:
        dict: The response from the API.
    """

    headers = {
        'Content-Type': 'application/json',  # To make a valid request it's necessary to state the correct 'Content-Type' (Note: in the other POST APIs [point 6 and 7: import codelist entries] there is not need to state the 'Content-Type', in fact stating it it could lead to errors)
        'User-Agent': "Political Parties (contact: benedikt.hitz@bfh.ch)",
        'Authorization': access_token
    }

    concept_template = copy.deepcopy(concept_template)  # Create a copy of the concept template to avoid modifying the original

    url = f'https://api-a.i14y.admin.ch/api/partner/v1/concepts/{current_concept["data"]["id"]}'

    concept_template["data"]["validFrom"] = current_concept["data"]["validFrom"]
    concept_template["data"]["validTo"] = end_date
    concept_template["data"]["version"] = current_concept["data"]["version"]

    response = requests.put(url, headers=headers, data=json.dumps(concept_template), verify=False)

    print(f'Status-Code: {response.status_code} - {response.text}')


In [8]:
def create_version_number(old_version_number, update = "MINOR"):
    """
    Create a new version number based on the old version number.

    Args:
        old_version_number (str): The old version number, e.g. "2024.6.1".
        update (str): The type of update, either "MAJOR", "MINOR", or "PATCH". Default is "MINOR".

    Returns:
        str: The new version number.
    """
    
    major, minor, patch = map(int, old_version_number.split('.'))

    if update == "MAJOR":
        major += 1
        minor = 0
        patch = 0
    elif update == "MINOR":
        minor += 1
        patch = 0
    elif update == "PATCH":
        patch += 1
    else:
        raise ValueError("Invalid update type. Must be 'MAJOR', 'MINOR', or 'PATCH'.")

    return f"{major}.{minor}.{patch}"

In [9]:
def create_concept(concept_template, access_token, valid_from, version):
    """
    Create a new concept in the i14y API.

    Args:
        concept_template (dict): The concept template on which to base the new concept.
        access_token (str): The access token.
        valid_from (str): The valid from date in isoformat, e.g. "2024-06-30T00:00:00".
        version (str): The version number according to semantic versioning, e.g. "1.6.0".

    Returns:
        str: The versioned identifier of the newly created concept.
    """

    headers = {
        'Content-Type': 'application/json',  # To make a valid request it's necessary to state the correct 'Content-Type' (Note: in the other POST APIs [point 6 and 7: import codelist entries] there is not need to state the 'Content-Type', in fact stating it it could lead to errors)
        'User-Agent': "Political Parties (contact: benedikt.hitz@bfh.ch)",
        'Authorization': access_token
    }

    concept_template = copy.deepcopy(concept_template)  # Create a copy of the concept template to avoid modifying the original

    url = 'https://api-a.i14y.admin.ch/api/partner/v1/concepts'

    # add the validFrom and version to the concept template
    concept_template["data"]["validFrom"] = valid_from
    concept_template["data"]["version"] = version

    response = requests.post(url, headers=headers, data=json.dumps(concept_template), verify=False)

    if response.status_code == 201:
        print(f'Status-Code: {response.status_code}')
        concept_id = response.json()
        return concept_id
    else:
        print(f"Error: {response.status_code} - {response.text}")

In [10]:
def add_concept_entries(versioned_identifier, concept_entries, access_token):
    """
    Add concept entries to a concept in the i14y API.

    Args:
        versioned_identifier (str): The versioned identifier of the concept.
        concept_entries (dict): The concept entries to add.
        access_token (str): The access token.
    """

    headers = { 
        'Authorization': access_token,
        'User-Agent': "Political Parties (contact: benedikt.hitz@bfh.ch)",
    }

    url = f'https://api-a.i14y.admin.ch/api/partner/v1/concepts/{versioned_identifier}/codelist-entries/imports/json'

    payload = json.dumps(concept_entries).encode('utf-8')

    files = {
        'file': ('i14_concept_entries.json', payload, 'application/json')
    }
    response = requests.post(url, headers=headers, files=files, verify = False)

    print(f'Status-Code: {response.status_code} - {response.text}')
    print(response)
        

In [11]:
def set_concept_to_public(versioned_identifier, access_token):
    """
    Set a concept to public in the i14y API.

    Args:
        versioned_identifier (str): The versioned identifier of the concept.
        access_token (str): The access token.
    """

    headers = { 
        'Authorization': access_token,
        'User-Agent': "Political Parties (contact: benedikt.hitz@bfh.ch)",
    }

    params = {
        'conceptId': versioned_identifier,
        'level': "Public" 
    }

    url = f'https://api-a.i14y.admin.ch/api/partner/v1/concepts/{versioned_identifier}/publication-level'


    response = requests.put(url, headers=headers, params=params)

    print(f'Status-Code: {response.status_code} - {response.text}')
    print(response)
    

In [12]:
def delete_concept(versioned_identifier, access_token):
    """
    Delete a concept in the i14y API.

    Args:
        versioned_identifier (str): The versioned identifier of the concept.
        access_token (str): The access token.
    """

    headers = { 
        'Authorization': access_token,
        'User-Agent': "Political Parties (contact: benedikt.hitz@bfh.ch)",
    }

    url = f"https://api-a.i14y.admin.ch/api/partner/v1/concepts/{versioned_identifier}"

    response = requests.delete(url, headers=headers, verify = False)

    if response.status_code == 204:
        print("Code List entries deleted successfully")
    else:
        print(f"Error: {response.status_code} - {response.text}")

## Run with No Existing Concept on I14Y

Initial Run that does not check for existing concepts on I14Y, but creates a new concept and uploads the concept entries to it.

In [ ]:
# read parties.json to dict
with open('../data/output/parties.json', 'r', encoding='utf-8') as file:
    parties = json.load(file)

new_concept_entries = create_concept_entries_from_json(parties)

# read concept_template.json to dict
with open('../data/output/concept_template.json', 'r', encoding='utf-8') as file:
    concept_template = json.load(file)

# set ENV
ENV = "INT"

# Load the env file
load_dotenv(f"../.env.{ENV.lower()}")

# get access token
access_token = get_access_token(os.getenv("I14Y_CLIENT_ID"), os.getenv("I14Y_CLIENT_SECRET"))

# get current date in isoformat
current_date = datetime.now().isoformat()

# define semantic versioning number
version = "1.0.0"

# create a new concept in i14y
versioned_identifier = create_concept(concept_template, access_token, current_date, version)

# upload the concept entries to the newly created concept
add_concept_entries(versioned_identifier, new_concept_entries, access_token)

# set the concept to public
set_concept_to_public(versioned_identifier, access_token)

## Run with Existing Concept on I14Y

In [15]:
# read parties.json to dict
with open('../data/output/parties.json', 'r', encoding='utf-8') as file:
    parties = json.load(file)

new_concept_entries = create_concept_entries_from_json(parties)

# read concept_template.json to dict
with open('../data/output/concept_template.json', 'r', encoding='utf-8') as file:
    concept_template = json.load(file)

# get the current concept from i14y
static_identifier = "political-parties"
current_concept = get_current_concept(static_identifier)

# get the current concept entries from i14y
current_concept_entries = get_current_concept_entries(current_concept["data"]["id"])

# compare the current concept entries with the new concept entries
if compare_concept_entries(current_concept_entries, new_concept_entries):
    
    print("Changes detected. Updating the concept on i14y.")

    # set ENV
    ENV = "INT"

    # Load the env file
    load_dotenv(f"../.env.{ENV.lower()}")

    # get access token
    access_token = get_access_token(os.getenv("I14Y_CLIENT_ID"), os.getenv("I14Y_CLIENT_SECRET"))

    # get current date in isoformat
    current_date = datetime.now().isoformat()
    
    # if there are changes, set the end date of the current concept to today
    set_concept_end_date(current_concept, concept_template, current_date, access_token)

    # create a new version number based on the old version number with "MINOR" update
    new_version = create_version_number(current_concept["data"]["version"], update = "MINOR")

    # create a new concept in i14y
    versioned_identifier = create_concept(concept_template, access_token, current_date, new_version)

    # upload the concept entries to the newly created concept
    add_concept_entries(versioned_identifier, new_concept_entries, access_token)

    # set the concept to public
    set_concept_to_public(versioned_identifier, access_token)

else:
    print("No changes detected. The concept on i14y is up to date.")

/Users/hib1/Nextcloud/Projekte/26_BK_Parties/fch-political-party-pipeline/.venv/lib/python3.13/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api-a.i14y.admin.ch'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Current concept: {'data': [{'codeListEntryDefaultSortProperty': 'Position', 'codeListEntryValueMaxLength': 30, 'codeListEntryValueType': 'String', 'conceptType': 'CodeList', 'conformsTo': [{'label': {'de': 'Parteien auf LINDAS', 'en': 'Parties on LINDAS', 'fr': 'Partis sur LINDAS', 'it': 'Partiti su LINDAS', 'rm': 'Partidas sin LINDAS'}, 'uri': 'https://politics.ld.admin.ch/political-party'}, {'label': {'de': 'Parteienatlas Année Politique Suisse', 'en': 'Party Atlas of Année Politique Suisse', 'fr': 'Atlas des partis d’Année Politique Suisse', 'it': 'Atlante dei partiti di Année Politique Suisse', 'rm': 'Partidatlas d’Année Politique Suisse'}, 'uri': 'https://parteienatlas.swiss'}], 'description': {'de': 'Die Codeliste enthält die Identifikatoren und Namen politischer Parteien in der Schweiz auf Bundes- und Kantonsebene. Die Namen insbesondere der Parteien auf nationaler Ebene sind in mehreren Landessprachen verfügbar. Die Liste ermöglicht eine einheitliche Codierung von Parteien in u

/Users/hib1/Nextcloud/Projekte/26_BK_Parties/fch-political-party-pipeline/.venv/lib/python3.13/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api-a.i14y.admin.ch'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Changes detected. The code list needs to be updated.
Changes detected. Updating the concept on i14y.


/Users/hib1/Nextcloud/Projekte/26_BK_Parties/fch-political-party-pipeline/.venv/lib/python3.13/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'identity.i14y.a.c.bfs.admin.ch'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/Users/hib1/Nextcloud/Projekte/26_BK_Parties/fch-political-party-pipeline/.venv/lib/python3.13/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api-a.i14y.admin.ch'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Status-Code: 204 - 


/Users/hib1/Nextcloud/Projekte/26_BK_Parties/fch-political-party-pipeline/.venv/lib/python3.13/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api-a.i14y.admin.ch'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Status-Code: 201


/Users/hib1/Nextcloud/Projekte/26_BK_Parties/fch-political-party-pipeline/.venv/lib/python3.13/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api-a.i14y.admin.ch'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Status-Code: 204 - 
<Response [204]>
Status-Code: 204 - 
<Response [204]>
